# Day 035 — Exercise 3: batch_extract

**What you'll build:** `async def batch_extract(docs, max_concurrent=3, model) -> list[dict]` — run `async_extract` concurrently on all docs using `asyncio.Semaphore` + `asyncio.gather`.

**Why it matters:** The concurrency layer. With `max_concurrent=3`, three LLM extractions run simultaneously — batching 10 documents takes roughly the same wall time as batching 3 serially. Check cells use top-level `await`.

## Provided: All Functions up to async_extract

In [ ]:
import requests
from pathlib import Path

def fetch_text(source: str) -> dict:
    src  = str(source)
    text = None
    kind = 'text'

    if src.startswith('http://') or src.startswith('https://'):
        try:
            response = requests.get(src, timeout=10)
            response.raise_for_status()
            text = response.text
            kind = 'url'
        except Exception as e:
            text = '[fetch error: ' + str(e) + ']'
            kind = 'url_error'
    else:
        try:
            p = Path(src)
            if p.exists() and p.is_file():
                text = p.read_text(encoding='utf-8')
                kind = 'file'
        except Exception:
            pass

    if text is None:
        text = src
        kind = 'text'

    return {
        'source':     src,
        'kind':       kind,
        'content':    text,
        'char_count': len(text),
    }


import json
import ollama
from pydantic import BaseModel, Field

class ArticleInfo(BaseModel):
    title:      str       = Field(description='Topic or title in 3-6 words')
    summary:    str       = Field(description='One sentence summary')
    sentiment:  str       = Field(description='positive, negative, or neutral')
    key_points: list[str] = Field(default_factory=list,
                                  description='Up to 3 key points as short phrases')

def extract_info(doc: dict, model: str = 'llama3.2') -> dict:
    schema = ArticleInfo.model_json_schema()
    prompt = (
        'Extract information from the document below. '
        'Return valid JSON matching this schema:\n'
        + json.dumps(schema, indent=2)
        + '\n\nDocument:\n' + doc['content'][:1500]
    )
    try:
        response = ollama.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            format='json',
        )
        info = ArticleInfo.model_validate_json(response['message']['content'])
        return {**doc, 'info': info.model_dump(), 'status': 'ok',    'error': None}
    except Exception as e:
        return {**doc, 'info': None,               'status': 'error', 'error': str(e)}


import asyncio
import json
import ollama

async def async_extract(doc: dict, model: str = 'llama3.2') -> dict:
    client = ollama.AsyncClient()
    schema = ArticleInfo.model_json_schema()
    prompt = (
        'Extract information from the document below. '
        'Return valid JSON matching this schema:\n'
        + json.dumps(schema, indent=2)
        + '\n\nDocument:\n' + doc['content'][:1500]
    )
    try:
        response = await client.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            format='json',
        )
        info = ArticleInfo.model_validate_json(response['message']['content'])
        return {**doc, 'info': info.model_dump(), 'status': 'ok',    'error': None}
    except Exception as e:
        return {**doc, 'info': None,               'status': 'error', 'error': str(e)}

## Your Implementation

In [ ]:
async def batch_extract(docs: list, max_concurrent: int = 3,
                        model: str = 'llama3.2') -> list[dict]:
    """
    Run async_extract concurrently on all docs with a semaphore.

    Returns a list of dicts of the same length as docs — each is
    the error envelope returned by async_extract.
    """
    # TODO: if not docs: return []
    # TODO: sem = asyncio.Semaphore(max_concurrent)
    # TODO: async def _run(doc):
    #     async with sem:
    #         return await async_extract(doc, model)
    # TODO: return list(await asyncio.gather(*[_run(d) for d in docs]))
    pass

## Check Your Work

In [ ]:
import asyncio

async def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined and is a coroutine function
    try:
        assert 'batch_extract' in globals()
        assert asyncio.iscoroutinefunction(batch_extract), \
            'batch_extract must be async def'
        passed += 1; print('\u2705 Check 1: batch_extract is async def')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: empty input
    try:
        result = await batch_extract([])
        assert result == [], f'expected [], got {result}'
        passed += 1; print('\u2705 Check 2: batch_extract([]) returns []')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Pre-run 2 docs for checks 3-5
    docs = [
        fetch_text('Python is a popular programming language for data science and AI.'),
        fetch_text('Machine learning uses statistical methods to find patterns in data.'),
    ]
    results = None
    try:
        results = await batch_extract(docs)
    except Exception as e:
        print(f'\u274c batch_extract call failed: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: result length == input length
    try:
        assert len(results) == 2, f'expected 2 results, got {len(results)}'
        passed += 1; print(f'\u2705 Check 3: 2 docs \u2192 2 results')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: each result has status key
    try:
        for r in results:
            assert 'status' in r, f'missing status key in result'
            assert r['status'] in ('ok', 'error'), \
                f"invalid status: {r['status']!r}"
        passed += 1; print('\u2705 Check 4: all results have status key')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: ok results have info dict
    try:
        ok = [r for r in results if r['status'] == 'ok']
        assert ok, 'no successful extractions (is Ollama running?)'
        for r in ok:
            info = r.get('info', {})
            assert isinstance(info, dict), f'info should be dict, got {type(info).__name__}'
            for k in ('title', 'summary', 'sentiment'):
                assert k in info, f'info missing: {k}'
        passed += 1; print(f'\u2705 Check 5: ok results have info dict '
                           f'(title, summary, sentiment) — {len(ok)}/{len(results)} ok')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


await _run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
async def batch_extract(docs: list, max_concurrent: int = 3,
                        model: str = 'llama3.2') -> list[dict]:
    if not docs:
        return []
    sem = asyncio.Semaphore(max_concurrent)
    async def _run(doc):
        async with sem:
            return await async_extract(doc, model)
    return list(await asyncio.gather(*[_run(d) for d in docs]))
```

</details>